這份 Notebook 是[《Deep Learning with Python》第三版](https://www.manning.com/books/deep-learning-with-python-third-edition)的範例。為方便閱讀，只保留可執行的程式碼與小節標題，不包含書中的說明文字、圖片與虛擬碼。

**建議搭配本書閱讀，才能掌握各段程式的用途與原理。**

作者公開英文版也可在 [deeplearningwithpython.io](https://deeplearningwithpython.io) 線上閱讀。

In [0]:
!pip install keras keras-hub --upgrade -q

In [0]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [0]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## 實務上的最佳實踐

### 讓模型發揮最大效用

#### 超參數最佳化

##### 使用 KerasTuner

In [0]:
!pip install keras-tuner -q

**程式 18.1 KerasTuner 模型建構函式**

In [0]:
import keras
from keras import layers

def build_model(hp):
    units = hp.Int(name="units", min_value=16, max_value=64, step=16)
    model = keras.Sequential(
        [
            layers.Dense(units, activation="relu"),
            layers.Dense(10, activation="softmax"),
        ]
    )
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

**程式 18.2 繼承 HyperModel 的子類別**

In [0]:
import keras_tuner as kt

class SimpleMLP(kt.HyperModel):
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def build(self, hp):
        units = hp.Int(name="units", min_value=16, max_value=64, step=16)
        model = keras.Sequential(
            [
                layers.Dense(units, activation="relu"),
                layers.Dense(self.num_classes, activation="softmax"),
            ]
        )
        optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model

hypermodel = SimpleMLP(num_classes=10)

In [0]:
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=20,
    executions_per_trial=2,
    directory="mnist_kt_test",
    overwrite=True,
)

In [0]:
tuner.search_space_summary()

In [0]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255
x_train_full = x_train[:]
y_train_full = y_train[:]
num_val_samples = 10000
x_train, x_val = x_train[:-num_val_samples], x_train[-num_val_samples:]
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:]
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
]
tuner.search(
    x_train,
    y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2,
)

**程式 18.3 查詢最佳的超參數配置**

In [0]:
top_n = 4
best_hps = tuner.get_best_hyperparameters(top_n)

In [0]:
def get_best_epoch(hp):
    model = build_model(hp)
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=10
        )
    ]
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=100,
        batch_size=128,
        callbacks=callbacks,
    )
    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [0]:
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(
        x_train_full, y_train_full, batch_size=128, epochs=int(best_epoch * 1.2)
    )
    return model

best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(x_test, y_test)
    best_models.append(model)

In [0]:
best_models = tuner.get_best_models(top_n)

##### 雕琢出正確搜尋空間的藝術

##### 超參數調整的未來：自動化機器學習

#### 模型集成（model ensembling）

### 使用多個設備擴大模型的訓練規模

#### 在多個 GPU 上訓練

##### 資料平行化：在每個 GPU 上複製模型

##### 模型平行化：將模型拆到多個 GPU 上

#### 分散式訓練實務

##### 取得兩個或更多的 GPU

##### 使用 JAX 進行資料平行化

##### 使用 JAX 進行模型平行化

###### DeviceMesh API

###### LayoutMap API

#### 在 TPU 上進行訓練

##### 利用步驟融合（step fusing）來提升 TPU 使用率

### 使用低精度運算加快訓練與推論

##### 認識浮點精度

##### float16 推論

##### 混合精度訓練

##### 混合精度的 loss scaling

##### 超越混合精度：float8 訓練

#### 使用量化加快推論

In [0]:
from keras import ops

x = ops.array([[0.1, 0.9], [1.2, -0.8]])
kernel = ops.array([[-0.1, -2.2], [1.1, 0.7]])

In [0]:
def abs_max_quantize(value):
    abs_max = ops.max(ops.abs(value), keepdims=True)
    scale = ops.divide(127, abs_max + 1e-7)
    scaled_value = value * scale
    scaled_value = ops.clip(ops.round(scaled_value), -127, 127)
    scaled_value = ops.cast(scaled_value, dtype="int8")
    return scaled_value, scale

int_x, x_scale = abs_max_quantize(x)
int_kernel, kernel_scale = abs_max_quantize(kernel)

In [0]:
int_y = ops.matmul(int_x, int_kernel)
y = ops.cast(int_y, dtype="float32") / (x_scale * kernel_scale)

In [0]:
y

In [0]:
ops.matmul(x, kernel)